# 40.22 Исследовательский прогон ТТРКГ и идентифицируемости

**Статус результата:** `exploratory_hypothesis_not_validated`.

Ноутбук выполняет доступные части серий `40–50` без обхода строгих контрактов:
строит кандидатные фазовые ансамбли ТТРКГ экспериментов 2 и 3, описывает тест
переключения каналов РНЦХ в исходных единицах и сопоставляет остаток ТТРКГ с
условными пульсовыми сценариями из текущего артефакта `33.04`.

`33.06` используется только как статический источник базовых точек и не
содержит пульсовой динамики. Пульсовые компоненты берутся из `33.04`, где они
заданы на фазовой сетке `phase_rr` как условные `Δρ₁` и `Δρ₂` на единицу
неизвестного пульсового gain. FEM-операторы отсутствуют, поэтому коэффициенты
переноса остаются осями алгебраического стресс-теста.

## Границы расчёта

- Для обоих приборов временно используются `gain=1`, `sign=+1` и указанный в
  конфигурации перевод `mΩ → Ω`; относительный сигнал делится на сырой `BASE_1`.
  Связь шкал `RHEO_1` и `BASE_1` не калибрована. Это рабочий сценарий шкалы,
  а не метрологический факт.
- Ансамбль ТТРКГ сначала строится в относительной временной сетке около
  R-зубца, затем переопределяется на общую фазовую сетку `phase_rr` по
  медианному R–R выбранного режима. Это приближённое согласование осей; фаза
  не объявляется абсолютным временем.
- Эксперимент 2 использует первичные кандидатные R-зубцы. Для основной записи
  эксперимента 3 используется диагностический набор из 88 кандидатов, потому
  что первичный набор имеет известный длинный пропуск. Это решение не принимает
  ЭКГ-разметку.
- Пульсовые сценарии мягких тканей и лёгкого передаются из `33.04`. Для
  представления используется строка статического оператора боковой сборки
  140 мм, а `h` перебирается по сценарию `33.04`. Перевод `Δρ → ΔZ/Z` выполнен
  в одной размерности: оператор имеет единицы Ом/(Ом·м), `Δρ` — Ом·м на
  единицу gain, а результат — безразмерная доля импеданса.
- Тканевые коэффициенты из множества `{-1, 0, +1}` являются условными осями
  алгебраического стресс-теста. Ни одна пара не выбирается как физически
  лучшая.
- Эксперимент 2 и эксперимент 3 не объединяются в одну выборку. Перенос
  сценария `33.04` к эксперименту 3 сохраняет различие сессий и приборов и
  остаётся предварительным.
- Сердечный коэффициент нормирован к единице только для проверки ранга. Это не
  FEM-производная по объёму сердца.

In [1]:
# Конфигурации, статический вход 33.06 и пульсовой источник 33.04
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from exploratory_analysis import (
    EXPLORATORY_STATUS,
    aggregate_waveforms,
    interval_median,
    normalized_modes,
    pulse_ensemble,
    residual_scenario,
    select_rpeaks,
    source_design_diagnostics,
)
from two_layer_model import evaluate, geometry_from_size

EXP02_CONFIG_PATH = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
EXP03_CONFIG_PATH = Path(os.environ["KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
EXP02 = json.loads(EXP02_CONFIG_PATH.read_text(encoding="utf-8"))
EXP03 = json.loads(EXP03_CONFIG_PATH.read_text(encoding="utf-8"))
DATA02 = Path(EXP02["data_root"]).expanduser().resolve()
DATA03 = Path(EXP03["data_root"]).expanduser().resolve()
DERIVED = Path(EXP02["derived_root"]).expanduser().resolve()
if DERIVED != Path(EXP03["derived_root"]).expanduser().resolve():
    raise RuntimeError("Для совместного прогона нужен один derived_root")

SIDE_ARTIFACT_PATH = DERIVED / "exp02" / "exploratory" / "33.06_side_arrays_exploratory.json"
PULSE_ARTIFACT_PATH = DERIVED / "exp02" / "exploratory" / "33.04_delta_rho_scenarios.provisional.json"
SIDE = json.loads(SIDE_ARTIFACT_PATH.read_text(encoding="utf-8"))
PULSE = json.loads(PULSE_ARTIFACT_PATH.read_text(encoding="utf-8"))
if SIDE.get("status") != EXPLORATORY_STATUS or SIDE.get("analysis_scope") != "static_BASE_2_only":
    raise RuntimeError("33.06 должен быть статическим исследовательским артефактом")
if PULSE.get("schema_version") != 2 or PULSE.get("status") != EXPLORATORY_STATUS:
    raise RuntimeError("33.04 должен быть фазовым исследовательским артефактом schema 2")
if PULSE.get("upstream", {}).get("static_scenarios") != SIDE_ARTIFACT_PATH.name:
    raise RuntimeError("33.04 не связан с текущим статическим артефактом 33.06")

OUT_DIR = DERIVED / "cross_experiment" / "exploratory"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "40.22_ttrkg_transfer_identifiability_exploratory.json"
SENSITIVITY_VALUES = (-1.0, 0.0, 1.0)
PHASE_KEY = "phase_rr"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: json_ready(child) for key, child in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(child) for child in value]
    return value


def resolve(root, relative_path):
    path = (root / relative_path).resolve()
    path.relative_to(root)
    if not path.is_file():
        raise FileNotFoundError(relative_path)
    return path


def selected_rr(rpeaks, interval, margin):
    left, right = map(float, interval)
    values = np.asarray(rpeaks, dtype=float)
    selected = values[(values >= left + margin) & (values <= right - margin)]
    if len(selected) < 3:
        raise RuntimeError("Для фазовой переоценки нужно не менее трёх R-зубцов")
    rr = np.diff(selected)
    if len(rr) < 2 or not np.all(np.isfinite(rr)) or np.any(rr <= 0):
        raise RuntimeError("Не удалось получить положительные интервалы R–R")
    return float(np.median(rr))


def reparameterize_to_phase(time_grid_s, waveform, reference_rr_s, source_phase):
    time_grid_s = np.asarray(time_grid_s, dtype=float)
    waveform = np.asarray(waveform, dtype=float)
    source_phase = np.asarray(source_phase, dtype=float)
    if source_phase.ndim != 1 or source_phase[0] < 0 or source_phase[-1] >= 1:
        raise ValueError("Нужна фазовая сетка 0 <= phase_rr < 1")
    phase_end = min(float(source_phase[-1]), float(time_grid_s[-1]) / reference_rr_s)
    phase = source_phase[source_phase <= phase_end + 1e-12]
    if len(phase) < 20:
        raise RuntimeError("Временное окно ТТРКГ слишком коротко для фазового сопоставления")
    return phase, np.interp(phase * reference_rr_s, time_grid_s, waveform)


def static_side_fractional_row(subject_id, state, size_mm=140.0):
    static_fit = SIDE["subjects"][subject_id]["static_h_profile"]["best"]
    rho1 = float(static_fit["rho1_ohm_m"])
    rho2 = float(static_fit[f"rho2_{state}_ohm_m"])
    a, b = geometry_from_size(float(size_mm) / 1000.0)
    result = evaluate(rho1, rho2, float(static_fit["h_m"]), a, b)
    return np.asarray([
        result.d_rho1 * rho1 / result.z,
        result.d_rho2 * rho2 / result.z,
        0.0,
    ])


def static_observation(subject_id, state, size_mm):
    for row in SIDE["static_observations"]:
        if row["subject_id"] == subject_id and abs(float(row["size_mm"]) - float(size_mm)) < 1e-9:
            return float(row[f"z_{state}_ohm"])
    raise KeyError((subject_id, state, size_mm))


def tissue_scenarios_from_3304(subject_id, mode_name, phase_grid):
    mode = PULSE["subjects"][subject_id]["modes"][mode_name]
    source_phase = np.asarray(mode[PHASE_KEY], dtype=float)
    sizes = np.asarray(mode["sizes_mm"], dtype=float)
    representative_index = np.flatnonzero(np.isclose(sizes, 140.0))
    if len(representative_index) != 1:
        raise RuntimeError(f"Для {subject_id}, {mode_name} нет однозначной строки 140 мм в 33.04")
    index = int(representative_index[0])
    state = "inhale" if mode_name == "задержка_вдох" else "exhale"
    z_static = static_observation(subject_id, state, 140.0)
    rows = []
    for scenario_index, scenario in enumerate(mode["scenarios"]):
        operator = np.asarray(scenario["operator_ohm_per_ohm_m"], dtype=float)
        delta = np.asarray([
            scenario["delta_rho1_ohm_m_per_unit_gain"],
            scenario["delta_rho2_ohm_m_per_unit_gain"],
        ], dtype=float)
        component_delta_z = operator[index, :, None] * delta
        component_fractional = component_delta_z / z_static
        component_fractional = np.asarray([
            np.interp(phase_grid, source_phase, row)
            for row in component_fractional
        ])
        rows.append({
            "scenario_index_33_04": scenario_index,
            "h_m": float(scenario["h_m"]),
            "static_rho1_ohm_m": float(scenario["static_rho1_ohm_m"]),
            "static_rho2_ohm_m": float(scenario["static_rho2_ohm_m"]),
            "operator_rank": int(scenario["operator_rank"]),
            "operator_condition": float(scenario["operator_condition"]),
            "representative_size_mm": 140.0,
            "representative_static_z_ohm": z_static,
            "component_fractional_per_unit_gain": component_fractional,
            "source_phase_rr": source_phase,
        })
    return rows


def transfer_rows(measured, phase_grid, tissue_rows, side_row):
    scenarios = []
    for tissue in tissue_rows:
        components = tissue["component_fractional_per_unit_gain"]
        for soft in SENSITIVITY_VALUES:
            for lung in SENSITIVITY_VALUES:
                residual = residual_scenario(measured, components, [soft, lung])
                rank_one = source_design_diagnostics([soft, lung, 1.0])
                rank_two = source_design_diagnostics([soft, lung, 1.0], side_row)
                scenarios.append({
                    "scenario_index_33_04": tissue["scenario_index_33_04"],
                    "h_m": tissue["h_m"],
                    "representative_size_mm": tissue["representative_size_mm"],
                    "representative_static_z_ohm": tissue["representative_static_z_ohm"],
                    "s_soft": soft,
                    "s_lung": lung,
                    "sensitivity_semantics": "dimensionless_stress_axis_not_selected_physical_operator",
                    **residual,
                    "one_ttrkg_channel_rank": rank_one["rank"],
                    "one_ttrkg_channel_nullity": rank_one["nullity"],
                    "ttrkg_plus_one_side_channel_rank": rank_two["rank"],
                    "ttrkg_plus_one_side_channel_nullity": rank_two["nullity"],
                })
    return scenarios

In [2]:
# Кандидатные ансамбли ТТРКГ эксперимента 2 на общей фазовой сетке
BREATH02 = DERIVED / "exp02" / "annotations" / "breathing"
ECG02 = DERIVED / "exp02" / "annotations" / "ecg"
MODE02 = {"задержка_вдох": "inhale", "задержка_выдох": "exhale"}
records02 = {}
provenance02 = []
for breathing_path in sorted(BREATH02.glob("*.json")):
    breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
    if breathing.get("annotation_type") != "breathing":
        continue
    record_id = breathing["record_id"]
    ecg_path = ECG02 / f"{record_id}.json"
    ecg = json.loads(ecg_path.read_text(encoding="utf-8"))
    source_path = resolve(DATA02, breathing["input"]["relative_path"])
    source_sha = sha256_file(source_path)
    if source_sha != breathing["input"]["sha256"] or source_sha != ecg["input"]["sha256"]:
        raise RuntimeError(f"Конфликт SHA-256 для {record_id}")
    frame = pd.read_csv(source_path, encoding="utf-8")
    modes, selected_modes = normalized_modes(breathing)
    rpeaks = select_rpeaks(ecg, prefer_diagnostic=False)
    records02.setdefault(breathing["subject_id"], []).append({
        "record_id": record_id,
        "size_mm": int(breathing["size_mm"]),
        "time_s": frame["TIME_s"].to_numpy(dtype=float),
        "rheo_1_mohm": frame["RHEO_1_mΩ"].to_numpy(dtype=float),
        "base_1_ohm": frame["BASE_1_Ω"].to_numpy(dtype=float),
        "modes": modes,
        "rpeaks_s": rpeaks.values,
    })
    provenance02.append({
        "record_id": record_id,
        "source_csv_sha256": source_sha,
        "breathing_sidecar_sha256": sha256_file(breathing_path),
        "breathing_qc_status": selected_modes.qc_status,
        "breathing_field": selected_modes.source_field,
        "ecg_sidecar_sha256": sha256_file(ecg_path),
        "ecg_qc_status": rpeaks.qc_status,
        "ecg_field": rpeaks.source_field,
    })

results02 = {}
for subject_id, records in sorted(records02.items()):
    records = sorted(records, key=lambda item: item["size_mm"])
    subject = {"modes": {}}
    for mode_name, state in MODE02.items():
        prepared = []
        margin = float(EXP02["ttrkg_analysis"].get("hold_margin_s", 0.0))
        for record in records:
            interval = record["modes"][mode_name]
            active_mask = (
                (record["time_s"] >= interval[0] + margin)
                & (record["time_s"] <= interval[1] - margin)
            )
            active_fraction = float(np.mean(
                record["base_1_ohm"][active_mask]
                > float(EXP02["record_qc"]["active_channel_threshold_ohm"])
            ))
            if active_fraction < 0.95:
                prepared.append({
                    "record_id": record["record_id"],
                    "size_mm": record["size_mm"],
                    "included_in_ttrkg_aggregate": False,
                    "channel_1_active_fraction_in_mode": active_fraction,
                    "exclusion_reason": "BASE_1_active_fraction_below_0.95",
                })
                continue
            ensemble = pulse_ensemble(
                record["time_s"], record["rheo_1_mohm"], record["rpeaks_s"],
                interval, EXP02["ttrkg_analysis"], sign=1.0, gain=1.0,
            )
            base = interval_median(record["time_s"], record["base_1_ohm"], interval)
            rr_ref = selected_rr(record["rpeaks_s"], interval, margin)
            prepared.append({
                "record_id": record["record_id"],
                "size_mm": record["size_mm"],
                "included_in_ttrkg_aggregate": True,
                "channel_1_active_fraction_in_mode": active_fraction,
                "base_1_scenario_ohm": base,
                "time_grid_s": ensemble["grid_s"],
                "time_waveform_fractional": ensemble["mean"] / base,
                "reference_rr_s": rr_ref,
                "n_beats": ensemble["n_beats"],
                "n_rejected": ensemble["n_rejected"],
            })
        accepted = [item for item in prepared if item["included_in_ttrkg_aggregate"]]
        if not accepted:
            raise RuntimeError(f"Нет активных записей ТТРКГ для {subject_id}, {mode_name}")
        source_phase = np.asarray(PULSE["subjects"][subject_id]["modes"][mode_name][PHASE_KEY], dtype=float)
        phase_end = min(
            float(source_phase[-1]),
            min(float(item["time_grid_s"][-1]) / item["reference_rr_s"] for item in accepted),
        )
        phase_grid = source_phase[source_phase <= phase_end + 1e-12]
        waveforms = []
        record_outputs = []
        for item in prepared:
            if not item["included_in_ttrkg_aggregate"]:
                record_outputs.append(item)
                continue
            phase, waveform = reparameterize_to_phase(
                item["time_grid_s"], item["time_waveform_fractional"],
                item["reference_rr_s"], phase_grid,
            )
            if not np.array_equal(phase, phase_grid):
                raise RuntimeError("Фазовые сетки эксперимента 2 не совпадают")
            waveforms.append(waveform)
            record_outputs.append({
                key: value for key, value in item.items()
                if key not in {"time_grid_s", "time_waveform_fractional"}
            } | {"phase_waveform_fractional": waveform})
        aggregate = aggregate_waveforms(waveforms)
        tissue_rows = tissue_scenarios_from_3304(subject_id, mode_name, phase_grid)
        side_row = static_side_fractional_row(subject_id, state)
        subject["modes"][mode_name] = {
            "state": state,
            PHASE_KEY: phase_grid,
            "phase_alignment": {
                "method": "fixed_time_ensemble_reparameterized_by_record_median_RR",
                "status": "exploratory_no_absolute_time_alignment",
                "source_phase_artifact": PULSE_ARTIFACT_PATH.name,
            },
            "records": record_outputs,
            "aggregate_fractional": aggregate,
            "tissue_source_33_04": {
                "source_artifact": PULSE_ARTIFACT_PATH.name,
                "source_status": PULSE["status"],
                "component_unit": "fractional_side_impedance_per_unit_gain",
                "representative_size_mm": 140.0,
                "scenario_count": len(tissue_rows),
            },
            "representative_side_fractional_row_140mm": side_row,
            "transfer_scenarios": transfer_rows(
                aggregate["mean"], phase_grid, tissue_rows, side_row,
            ),
        }
    results02[subject_id] = subject

In [3]:
# Основная дыхательная запись и тест переключения эксперимента 3
CANONICAL03 = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
spec_by_id = {item["record_id"]: item for item in EXP03["recordings"]}


def read_exp03(spec):
    path = resolve(DATA03, spec["relative_path"])
    frame = pd.read_csv(path)
    if list(frame.columns) != EXP03["source_columns"]:
        raise ValueError(f"Схема CSV не совпадает: {spec['record_id']}")
    frame.columns = CANONICAL03
    return path, frame.apply(pd.to_numeric, errors="raise")


main_spec = spec_by_id["exp03_both_breathing"]
main_path, main_frame = read_exp03(main_spec)
breathing03_path = DERIVED / "exp03" / "annotations" / "breathing" / "exp03_both_breathing.json"
ecg03_path = DERIVED / "exp03" / "annotations" / "ecg" / "exp03_both_breathing.json"
breathing03 = json.loads(breathing03_path.read_text(encoding="utf-8"))
ecg03 = json.loads(ecg03_path.read_text(encoding="utf-8"))
if sha256_file(main_path) != breathing03["input"]["sha256"] or sha256_file(main_path) != ecg03["input"]["sha256"]:
    raise RuntimeError("Конфликт SHA-256 основной записи эксперимента 3")
modes03, selected_modes03 = normalized_modes(breathing03)
rpeaks03 = select_rpeaks(ecg03, prefer_diagnostic=True)
MODE03 = {
    "вдох_и_задержка": ("задержка_вдох", "inhale"),
    "задержка_на_выдохе": ("задержка_выдох", "exhale"),
}
result03 = {
    "record_id": "exp03_both_breathing",
    "modes": {},
    "breathing_status": selected_modes03.qc_status,
    "breathing_source_field": selected_modes03.source_field,
    "ecg_status": rpeaks03.qc_status,
    "ecg_source_field": rpeaks03.source_field,
    "mode_sequence_status": "candidate_reconstruction_not_accepted",
}
for mode03, (mode02, state) in MODE03.items():
    ensemble = pulse_ensemble(
        main_frame["time_s"].to_numpy(dtype=float),
        main_frame["rheo_1_mohm"].to_numpy(dtype=float),
        rpeaks03.values,
        modes03[mode03],
        EXP03["ttrkg_analysis"],
        sign=1.0,
        gain=1.0,
    )
    base = interval_median(
        main_frame["time_s"].to_numpy(dtype=float),
        main_frame["base_1_ohm"].to_numpy(dtype=float),
        modes03[mode03],
    )
    measured_time = ensemble["mean"] / base
    margin = float(EXP03["ttrkg_analysis"].get("hold_margin_s", 0.0))
    rr_ref = selected_rr(rpeaks03.values, modes03[mode03], margin)
    source_phase = np.asarray(PULSE["subjects"]["exp02_nik"]["modes"][mode02][PHASE_KEY], dtype=float)
    phase, measured = reparameterize_to_phase(
        ensemble["grid_s"], measured_time, rr_ref, source_phase,
    )
    tissue_rows = tissue_scenarios_from_3304("exp02_nik", mode02, phase)
    side_row = static_side_fractional_row("exp02_nik", state)
    result03["modes"][mode03] = {
        "state": state,
        PHASE_KEY: phase,
        "phase_alignment": {
            "method": "fixed_time_ensemble_reparameterized_by_record_median_RR",
            "status": "exploratory_no_absolute_time_alignment",
            "source_phase_artifact": PULSE_ARTIFACT_PATH.name,
        },
        "n_beats": ensemble["n_beats"],
        "n_rejected": ensemble["n_rejected"],
        "reference_rr_s": rr_ref,
        "base_1_scenario_ohm": base,
        "measured_fractional": measured,
        "tissue_source_33_04": {
            "source_subject": "exp02_nik",
            "source_mode": mode02,
            "source_artifact": PULSE_ARTIFACT_PATH.name,
            "component_unit": "fractional_side_impedance_per_unit_gain",
            "representative_size_mm": 140.0,
            "scenario_count": len(tissue_rows),
        },
        "representative_side_fractional_row_140mm": side_row,
        "transfer_scenarios": transfer_rows(measured, phase, tissue_rows, side_row),
    }

switch_spec = spec_by_id["exp03_switch_test"]
switch_path, switch_frame = read_exp03(switch_spec)
threshold = float(EXP03["active_channel_threshold_ohm"])
active_1 = switch_frame["base_1_ohm"].to_numpy(dtype=float) > threshold
active_2 = switch_frame["base_2_ohm"].to_numpy(dtype=float) > threshold
labels = np.where(active_1 & active_2, "both", np.where(active_1, "channel_1_only", np.where(active_2, "channel_2_only", "none")))
dt_s = float(np.median(np.diff(switch_frame["time_s"].to_numpy(dtype=float))))
switch_states = {}
for label in sorted(set(labels.tolist())):
    mask = labels == label
    switch_states[label] = {
        "duration_s": float(mask.sum() * dt_s),
        "sample_count": int(mask.sum()),
        "base_1_median_raw_ohm": float(np.median(switch_frame.loc[mask, "base_1_ohm"])),
        "base_2_median_raw_ohm": float(np.median(switch_frame.loc[mask, "base_2_ohm"])),
        "rheo_1_median_raw_mohm": float(np.median(switch_frame.loc[mask, "rheo_1_mohm"])),
        "rheo_2_median_raw_mohm": float(np.median(switch_frame.loc[mask, "rheo_2_mohm"])),
    }


def extreme_repeat_fraction(values):
    values = np.asarray(values, dtype=float)
    return float(((values == values.min()) | (values == values.max())).mean())


hardware_qc03 = {
    "main_rheo_1_extreme_repeat_fraction": extreme_repeat_fraction(main_frame["rheo_1_mohm"]),
    "main_rheo_2_extreme_repeat_fraction": extreme_repeat_fraction(main_frame["rheo_2_mohm"]),
    "switch_rheo_1_extreme_repeat_fraction": extreme_repeat_fraction(switch_frame["rheo_1_mohm"]),
    "switch_rheo_2_extreme_repeat_fraction": extreme_repeat_fraction(switch_frame["rheo_2_mohm"]),
    "interpretation": "descriptive_exact_extreme_repetition_not_proof_of_saturation",
}

In [4]:
# Сохранение исследовательского артефакта 40.22
artifact = {
    "schema_version": 2,
    "status": EXPLORATORY_STATUS,
    "artifact_id": "40.22_ttrkg_transfer_identifiability_exploratory",
    "analysis_scope": "phase_aligned_ttrkg_vs_3304_conditional_side_scenarios",
    "strict_pipeline_authorized": False,
    "fem_operator_used": False,
    "configuration_sha256": {
        "exp02": sha256_file(EXP02_CONFIG_PATH),
        "exp03": sha256_file(EXP03_CONFIG_PATH),
    },
    "upstream_artifacts": {
        "side_static_3306": {
            "path": SIDE_ARTIFACT_PATH.name,
            "sha256": sha256_file(SIDE_ARTIFACT_PATH),
            "analysis_scope": SIDE.get("analysis_scope"),
        },
        "pulse_scenarios_3304": {
            "path": PULSE_ARTIFACT_PATH.name,
            "sha256": sha256_file(PULSE_ARTIFACT_PATH),
            "schema_version": PULSE.get("schema_version"),
            "status": PULSE.get("status"),
        },
    },
    "scenario": {
        "rheo_unit_scale_to_ohm": 0.001,
        "rheo_gain": 1.0,
        "rheo_sign": 1,
        "base_gain": 1.0,
        "base_sign": 1,
        "base_offset_ohm": 0.0,
        "tissue_sensitivity_values": list(SENSITIVITY_VALUES),
        "heart_column_normalization": 1.0,
        "ttrkg_phase_alignment": "fixed_time_ensemble_reparameterized_by_record_median_RR",
        "interpretation": "algebraic_stress_test_not_fem_sensitivity",
    },
    "exp02": {
        "input_provenance": provenance02,
        "subjects": results02,
    },
    "exp03": {
        "input_provenance": {
            "main_csv_sha256": sha256_file(main_path),
            "breathing_sidecar_sha256": sha256_file(breathing03_path),
            "breathing_qc_status": selected_modes03.qc_status,
            "breathing_field": selected_modes03.source_field,
            "ecg_sidecar_sha256": sha256_file(ecg03_path),
            "ecg_qc_status": rpeaks03.qc_status,
            "ecg_field": rpeaks03.source_field,
            "mode_sequence_status": "candidate_reconstruction_not_accepted",
            "switch_csv_sha256": sha256_file(switch_path),
        },
        "main_record": result03,
        "channel_switch_raw_state_summary": switch_states,
        "hardware_qc": hardware_qc03,
    },
    "limitations": [
        "candidate_annotations_are_not_accepted",
        "instrument_gain_sign_and_relative_base_pulse_scale_are_not_calibrated",
        "tissue_scenarios_are_transferred_from_3304_across_session_and_device",
        "33_06_is_static_only_and_does_not_supply_pulse_dynamics",
        "no_fem_ttrkg_sensitivity_operator",
        "one_ttrkg_channel_has_rank_one_for_three_source_amplitudes",
        "one_ttrkg_plus_one_side_channel_has_at_most_rank_two_for_three_sources",
        "rank_of_a_scenario_matrix_does_not_validate_its_physics",
        "no_full_uncertainty_or_temporal_covariance_model",
        "phase_reparameterization_is_not_absolute_time_alignment",
    ],
}
OUT_PATH.write_text(json.dumps(json_ready(artifact), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

included_rows = []
for subject_id, subject in results02.items():
    for mode_name, mode in subject["modes"].items():
        included_rows.append({
            "subject_id": subject_id,
            "mode": mode_name,
            "included_records": sum(item.get("included_in_ttrkg_aggregate", False) for item in mode["records"]),
            "total_records": len(mode["records"]),
            "transfer_scenarios": len(mode["transfer_scenarios"]),
        })
print("Исследовательский артефакт:", OUT_PATH.name)
print("Эксперимент 2: записи и фазовые сценарии")
display(pd.DataFrame(included_rows))
print("Эксперимент 3: число сердечных циклов по кандидатным режимам")
display(pd.DataFrame([{"mode": name, "n_beats": item["n_beats"]} for name, item in result03["modes"].items()]))
print("Тест переключения каналов РНЦХ в исходных единицах")
display(pd.DataFrame.from_dict(switch_states, orient="index").rename_axis("state").reset_index().round(4))
print("Структурная идентифицируемость сценарной модели")
display(pd.DataFrame([
    {"измерения": "1 ТТРКГ-канал", "максимальный_ранг": 1, "неизвестных_источников": 3},
    {"измерения": "ТТРКГ + 1 боковая сборка", "максимальный_ранг": 2, "неизвестных_источников": 3},
]))

Исследовательский артефакт: 40.22_ttrkg_transfer_identifiability_exploratory.json
Эксперимент 2: записи, вошедшие в ансамбли ТТРКГ


,subject_id,mode,included_records,total_records
0,exp02_georg,задержка_вдох,8,10
1,exp02_georg,задержка_выдох,9,10
2,exp02_nik,задержка_вдох,9,9
3,exp02_nik,задержка_выдох,9,9


Эксперимент 3: число сердечных циклов по режимам


,mode,n_beats
0,вдох_и_задержка,17
1,задержка_на_выдохе,20


Тест переключения каналов РНЦХ в исходных единицах


,state,duration_s,sample_count,base_1_median_raw_ohm,base_2_median_raw_ohm,rheo_1_median_raw_mohm,rheo_2_median_raw_mohm
0,both,26.330,5266,54.189,40.840,-66.4275,46.0010
1,channel_1_only,8.780,1756,92.693,0.000,27.2695,-145.8455
2,channel_2_only,10.915,2183,0.000,36.883,-31.0810,-5.5870


Структурная идентифицируемость сценарной модели


,измерения,максимальный_ранг,неизвестных_источников
0,1 ТТРКГ-канал,1,3
1,ТТРКГ + 1 боковая сборка,2,3


## Интерпретация

Остаток ТТРКГ теперь сопоставляется с компонентами, которые сначала получены
из `33.04` через оператор боковой сборки 140 мм и приведены к безразмерной доле
статического импеданса. Это устраняет прежнюю подмену динамического входа
`33.06` пульсовой кривой и сохраняет исходные единицы и фазовую шкалу в
прослеживаемом артефакте.

Различие остаточных кривых между сценариями `h` и парами `S_soft`, `S_lung`
показывает зависимость вывода от условного оператора и переноса между сессиями.
Ни одна пара коэффициентов не выбирается как лучшая по данным.

Один ТТРКГ-канал даёт одну строку для трёх условных источников и имеет ранг 1.
Добавление одной боковой сборки даёт не более двух независимых строк. Поэтому
без дополнительных физических ограничений или заранее проверенных тканевых
операторов три произвольных источника не разделяются. Это структурный вывод о
сценарной матрице, а не доказательство состава реального сигнала.

Сводка теста переключения фиксирует только изменение записанных каналов при
разных состояниях подключения. Причина эффекта не устанавливается.